In [8]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML # Para mostrar la animación en Jupyter/Colab

# --- 1. Definición de la Geometría y Parámetros ---

LONGITUD = 10.0  # Longitud de la tubería
ANCHO_INICIAL = 2.0  # Ancho en el punto de entrada
ANCHO_FINAL = 0.5    # Ancho en el punto de salida
VELOCIDAD_INICIAL = 1.0  # Velocidad uniforme en la entrada (V1)

# Parámetros de la simulación de partículas
NUM_PARTICULAS = 50
TIEMPO_TOTAL = 5.0     # Duración total de la animación en segundos
DT = 0.05              # Paso de tiempo para la integración (segundos)
NUM_FRAMES = int(TIEMPO_TOTAL / DT)

# Color de las partículas para mejor visualización
PARTICLE_COLOR = 'cyan'
PIPE_COLOR = 'k'
PARTICLE_SIZE = 10 # Tamaño de los puntos que representan las partículas

# --- 2. Función para obtener el ancho de la tubería (CORREGIDA) ---

def get_pipe_width(x):
    # La pendiente de reducción lineal
    pendiente = (ANCHO_FINAL - ANCHO_INICIAL) / LONGITUD
    
    # La ecuación de la línea (ancho en la sección principal)
    ancho_calculado = ANCHO_INICIAL + pendiente * x
    
    # np.where aplica la lógica condicional elemento por elemento:
    # Si la posición x está fuera del dominio (0 a LONGITUD), 
    # usa el ancho inicial (ANCHO_INICIAL). 
    # De lo contrario, usa el ancho calculado (ancho_calculado).
    
    width = np.where(
        (x < 0) | (x > LONGITUD), # Condición: OR (|) lógico para arrays
        ANCHO_INICIAL,           # Valor si la condición es True (fuera del rango)
        ancho_calculado          # Valor si la condición es False (dentro del rango)
    )
    return width
# --- 3. Función para obtener el campo de velocidad ---

def get_velocity(x, y):
    """
    Calcula el vector de velocidad (Vx, Vy) en un punto (x, y)
    basado en la Ecuación de Continuidad para un flujo ideal.
    """
    current_width = get_pipe_width(x)
    
    # Si la partícula está fuera del ancho de la tubería, la velocidad es 0
    if abs(y) > current_width / 2:
        return 0.0, 0.0
    
    # Velocidad en X (por Continuidad: V(x) = V1 * (A1 / A(x)))
    # En 2D, el "área" es el ancho.
    vx = VELOCIDAD_INICIAL * (ANCHO_INICIAL / current_width)
    
    # Velocidad en Y (asumimos Vy = 0 para este flujo 2D ideal y horizontal)
    vy = 0.0
    
    return vx, vy

# --- 4. Inicialización de Partículas ---

# Posiciones iniciales de las partículas (dentro del ancho inicial)
# Distribuimos las partículas uniformemente en Y en la entrada.
initial_x = np.zeros(NUM_PARTICULAS)
initial_y = np.linspace(-ANCHO_INICIAL / 2 * 0.9, ANCHO_INICIAL / 2 * 0.9, NUM_PARTICULAS) # Un poco dentro de los bordes

# Almacenamos las posiciones de las partículas para cada fotograma
particle_positions = np.zeros((NUM_FRAMES, NUM_PARTICULAS, 2))
particle_positions[0, :, 0] = initial_x
particle_positions[0, :, 1] = initial_y

In [10]:
# --- 5. Simulación del Movimiento de Partículas (Integración Euler) ---
for i in range(1, NUM_FRAMES):
    current_x = particle_positions[i-1, :, 0]
    current_y = particle_positions[i-1, :, 1]
    
    new_x = np.copy(current_x)
    new_y = np.copy(current_y)
    
    for j in range(NUM_PARTICULAS):
        vx, vy = get_velocity(current_x[j], current_y[j])
        
        # Integración simple de Euler: P_nueva = P_vieja + V * dt
        new_x[j] += vx * DT
        new_y[j] += vy * DT
        
        # Reaparecer partículas en la entrada si salen por el final
        if new_x[j] > LONGITUD:
            new_x[j] = initial_x[j] # Vuelve al inicio
            new_y[j] = initial_y[j] # Manteniendo su posición Y inicial relativa
            
        # Opcional: Rebotar partículas en las paredes (si el flujo no es perfecto)
        # current_width = get_pipe_width(new_x[j])
        # if abs(new_y[j]) > current_width / 2:
        #     new_y[j] = np.clip(new_y[j], -current_width/2, current_width/2)
        #     # Si quieres que reboten, necesitarías invertir vy, pero aquí vy=0

    particle_positions[i, :, 0] = new_x
    particle_positions[i, :, 1] = new_y

# --- 6. Configuración de la Animación ---

fig, ax = plt.subplots(figsize=(10, 5))

# Dibujar las paredes de la tubería
x_pipe = np.linspace(0, LONGITUD, 100)
y_upper_pipe = get_pipe_width(x_pipe) / 2
y_lower_pipe = -get_pipe_width(x_pipe) / 2
ax.plot(x_pipe, y_upper_pipe, PIPE_COLOR, linewidth=3)
ax.plot(x_pipe, y_lower_pipe, PIPE_COLOR, linewidth=3)

# Inicializar las partículas para la animación
# scatter devuelve un objeto PathCollection
particles_plot, = ax.plot([], [], 'o', color=PARTICLE_COLOR, markersize=PARTICLE_SIZE, alpha=0.8)

ax.set_xlim(0, LONGITUD)
ax.set_ylim(-ANCHO_INICIAL / 2 - 0.5, ANCHO_INICIAL / 2 + 0.5)
ax.set_aspect('equal', adjustable='box')
ax.set_title("Animación de Flujo de Partículas en Tubería Estrecha (Fluido Ideal)")
ax.set_xlabel("Posición (m)")
ax.set_ylabel("Ancho (m)")
ax.grid(True, linestyle='--', alpha=0.6)

def animate(i):
    # Actualizar las posiciones de las partículas para el fotograma actual
    particles_plot.set_data(particle_positions[i, :, 0], particle_positions[i, :, 1])
    return particles_plot, # La coma es importante para el método blit

# Crear la animación
ani = animation.FuncAnimation(
    fig, animate, frames=NUM_FRAMES, blit=True, interval=DT*1000, repeat=True
)

# Mostrar la animación
plt.close() # Cierra la figura estática, solo queremos la animación

# Para mostrar en Jupyter Notebook / Google Colab:
HTML(ani.to_jshtml())

# Para guardar como MP4 (requiere ffmpeg instalado):
# ani.save('flujo_tubería_ideal.mp4', writer='ffmpeg', fps=1/DT)